# Phase 04 — Mechanism-Gate Experiment (Kaggle T4)

Runs the σ-Trap **mechanism gate**: 4 arms × n=15 × 2000 steps on the H-Bar
compositional benchmark, recording the **measured** Stage-1 proxy (GCA + RGA)
at every eval point. Logic lives in `sigma_align` (this notebook is thin
orchestration, per ADR-0004); parameters come from `configs/gate.yaml`.

## How to import `sigma_align`
1. Zip `code/` (or the repo root) and upload it as a **private Kaggle Dataset**
   (e.g. `/kaggle/input/sigma-model-code/`). Cell 1 auto-locates the package via
   `glob('/kaggle/input/**/sigma_align')`.
2. Dependencies: torch, numpy, scipy, pandas, matplotlib, seaborn, tqdm — all
   preinstalled on Kaggle.

## Expected runtime (T4)
- Default (`eval_every=25`, `eval_max_batches=8`): **≈ 6–8 h** (60 runs).
  The first run prints the per-run time + projected total — stop early and
  adjust if it exceeds the 9 h session budget.
- Faster: set `EVAL_EVERY = 50` (→ ≈ 4–5 h, coarser early resolution) or
  `EVAL_MAX_BATCHES = 4` (→ ≈ 5–6 h, noisier per-eval accuracy).

## Outputs (in `/kaggle/working/gate_output/` — download all)
- `all_results.pkl` — `{condition: [run metrics…]}`. `metrics.sigma_tilde` is the
  **measured** proxy (GCA+RGA fused); `metrics.sigma_sched` is the scheduled knob
  (diagnostic only — it does not lead OOD, archived finding 2026-08-16).
- `summary.json` — per-condition final means/std.
- `<condition>_run<id>.pkl` — per-run metrics (proxy trajectories).
- `trajectories.csv` — long-form trajectories for `analyze_gate.py`.


In [ ]:
# =============================================================================
# CELL 1: Setup — imports, paths, package import, reproducibility (CC.2.1)
# =============================================================================
import glob
import json
import os
import pickle
import random
import sys
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR = "/kaggle/working" if os.path.isdir("/kaggle") else "./output"
OUT_DIR = f"{BASE_DIR}/gate_output"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Package path ───────────────────────────────────────────────────────────
sys.path.insert(0, os.path.abspath("."))
if os.path.isdir("/kaggle"):
    _pkg_paths = glob.glob("/kaggle/input/**/sigma_align", recursive=True)
    if _pkg_paths:
        sys.path.insert(0, os.path.dirname(_pkg_paths[0]))
        print("sigma_align from:", _pkg_paths[0])

from sigma_align.config import load_config  # noqa: E402
from sigma_align.experiments.hbar_data import (  # noqa: E402
    generate_hard_compositional_data,
    make_loaders,
)
from sigma_align.experiments.hbar_train import (  # noqa: E402
    persist_run,
    train_hbar_model,
)

# ── Config (single source of truth: configs/gate.yaml) ─────────────────────
_cfg_paths = glob.glob("/kaggle/input/**/configs/gate.yaml", recursive=True)
cfg = load_config(_cfg_paths[0] if _cfg_paths else "code/experiments/configs/gate.yaml")
print("config:", _cfg_paths[0] if _cfg_paths else "code/experiments/configs/gate.yaml")

# ── Overrides (edit here only if needed; None = use gate.yaml) ─────────────
OVERRIDE_N_RUNS = None
OVERRIDE_N_TIMESTEPS = None
OVERRIDE_EVAL_EVERY = None
OVERRIDE_EVAL_MAX_BATCHES = None

GLOBAL_SEED = int(cfg["reproducibility"]["global_seed"])
N_RUNS = OVERRIDE_N_RUNS or int(cfg["experiment"]["n_runs_per_condition"])
N_TIMESTEPS = OVERRIDE_N_TIMESTEPS or int(cfg["training"]["n_timesteps"])
EVAL_EVERY = OVERRIDE_EVAL_EVERY or int(cfg["training"]["eval_every"])
EVAL_MAX_BATCHES = OVERRIDE_EVAL_MAX_BATCHES or int(cfg["training"]["eval_max_batches"])
CONDITIONS = list(cfg["experiment"]["conditions"])

# ── Reproducibility ────────────────────────────────────────────────────────
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB | AMP={USE_AMP}")
else:
    print("No GPU found — running on CPU")
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | "
      f"{datetime.now().strftime('%Y-%m-%d %H:%M')}")


In [ ]:
# =============================================================================
# CELL 2: Benchmark generation + loaders (pilot cells 1 & 3)
# =============================================================================
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

train_pairs, test_pairs, ood_pairs, comp_pairs, vocab = generate_hard_compositional_data(
    n_train=int(cfg["data"]["n_train"]),
    n_test_id=int(cfg["data"]["n_test_id"]),
    n_test_ood=int(cfg["data"]["n_test_ood"]),
    n_comp=int(cfg["data"]["n_comp"]),
)
loaders = make_loaders(
    train_pairs,
    test_pairs,
    ood_pairs,
    comp_pairs,
    vocab,
    batch_size=int(cfg["training"]["batch_size"]),
    seed=GLOBAL_SEED,
    num_workers=0 if not torch.cuda.is_available() else int(cfg["training"]["num_workers"]),
)
print(f"train: {len(loaders['train'].dataset)}  id: {len(loaders['id'].dataset)}  "
      f"ood: {len(loaders['ood'].dataset)}  comp: {len(loaders['comp'].dataset)}  "
      f"vocab: {len(vocab)}")


In [ ]:
# =============================================================================
# CELL 3: Run the gate — 4 arms × N_RUNS × N_TIMESTEPS (pilot cell 5)
# =============================================================================
all_results = {}

for condition in CONDITIONS:
    print(f"\n{'='*60}\nCONDITION: {condition.upper()}\n{'='*60}")
    cond_runs = []
    for run_id in range(N_RUNS):
        t0 = time.time()
        metrics = train_hbar_model(
            condition=condition,
            run_id=run_id,
            cfg=cfg,
            loaders=loaders,
            device=DEVICE,
            use_amp=USE_AMP,
            n_timesteps=N_TIMESTEPS,
            eval_every=EVAL_EVERY,
            lr=float(cfg["training"]["lr"]),
            save_checkpoints=False,
        )
        persist_run(metrics, condition, run_id, N_TIMESTEPS, OUT_DIR)
        cond_runs.append(metrics)
        dt = time.time() - t0
        if run_id == 0 and condition == CONDITIONS[0]:
            projected = dt * N_RUNS * len(CONDITIONS) / 3600
            print(f"  [timing] first run {dt/60:.1f} min → projected total ≈ {projected:.1f} h")
    all_results[condition] = cond_runs
    ood = [r["final"]["acc_ood"] for r in cond_runs]
    ida = [r["final"]["acc_id"] for r in cond_runs]
    print(f"{condition.upper():14s} ID={np.mean(ida):.1f}+-{np.std(ida):.1f}%  "
          f"OOD={np.mean(ood):.1f}+-{np.std(ood):.1f}%  gap={np.mean(ida)-np.mean(ood):.1f}%")

# ── Persist: all_results.pkl + summary.json (pilot cells 5 & 8 schema) ─────
with open(f"{OUT_DIR}/all_results.pkl", "wb") as f:
    pickle.dump(all_results, f)

summary_stats = {
    c: {
        "condition": c,
        "final_acc_id_mean": float(np.mean([r["final"]["acc_id"] for r in runs])),
        "final_acc_id_std": float(np.std([r["final"]["acc_id"] for r in runs])),
        "final_acc_ood_mean": float(np.mean([r["final"]["acc_ood"] for r in runs])),
        "final_acc_ood_std": float(np.std([r["final"]["acc_ood"] for r in runs])),
        "raw_acc_ood": [r["final"]["acc_ood"] for r in runs],
        "n_runs": len(runs),
    }
    for c, runs in all_results.items()
}
best_cond = max(summary_stats, key=lambda c: summary_stats[c]["final_acc_ood_mean"])
summary_json = {
    "experiment": {
        "date": datetime.now().isoformat(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "runs": sum(len(v) for v in all_results.values()),
        "amp": USE_AMP,
        "global_seed": GLOBAL_SEED,
    },
    "results": {
        c: {
            "mean_ood": summary_stats[c]["final_acc_ood_mean"],
            "std_ood": summary_stats[c]["final_acc_ood_std"],
            "mean_id": summary_stats[c]["final_acc_id_mean"],
            "n_runs": summary_stats[c]["n_runs"],
        }
        for c in summary_stats
    },
    "best_condition": best_cond,
    "compositional_gap": (
        summary_stats["baseline"]["final_acc_id_mean"]
        - summary_stats["baseline"]["final_acc_ood_mean"]
    ),
}
with open(f"{OUT_DIR}/summary.json", "w") as f:
    json.dump(summary_json, f, indent=2)

# ── Per-run proxy trajectories (long-form CSV for analysis) ────────────────
rows = []
for c, runs in all_results.items():
    for rid, r in enumerate(runs):
        for i, step in enumerate(r["step"]):
            rows.append({
                "condition": c,
                "run": rid,
                "step": step,
                "acc_id": r["acc_id"][i],
                "acc_ood": r["acc_ood"][i],
                "sigma_tilde": r["sigma_tilde"][i],
                "gca": r["gca"][i],
                "rga": r["rga"][i],
                "sigma_sched": r["sigma_sched"][i],
                "phase": r["phase"][i],
                "param_norm": r["param_norm"][i],
            })
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/trajectories.csv", index=False)

print(f"\nAll {sum(len(v) for v in all_results.values())} runs complete — saved to {OUT_DIR}")
print("files: all_results.pkl, summary.json, trajectories.csv, <condition>_run<id>.pkl")


In [ ]:
# =============================================================================
# CELL 4: Quick look — final OOD + measured σ̃_A trajectories (mean ± std)
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
conds = list(summary_stats)
ood_mu = [summary_stats[c]["final_acc_ood_mean"] for c in conds]
ood_sd = [summary_stats[c]["final_acc_ood_std"] for c in conds]
bars = ax.bar(conds, ood_mu, yerr=ood_sd, capsize=5, edgecolor="black")
for bar, mu in zip(bars, ood_mu):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f"{mu:.1f}%", ha="center", va="bottom", fontweight="bold")
ax.set(xlabel="Condition", ylabel="Final OOD Accuracy (%)",
       title="Mechanism Gate — Final OOD by Condition")

ax = axes[1]
for cname in conds:
    seqs = [r["sigma_tilde"] for r in all_results[cname]]
    steps = all_results[cname][0]["step"]
    ml = min(len(s) for s in seqs)
    mu = np.mean([s[:ml] for s in seqs], axis=0)
    sd = np.std([s[:ml] for s in seqs], axis=0)
    ax.plot(steps[:ml], mu, label=cname, lw=2)
    ax.fill_between(steps[:ml], mu - sd, mu + sd, alpha=0.15)
ax.set(xlabel="Training Step", ylabel="σ̃_A (measured proxy)", ylim=(0, 1),
       title="Measured Stage-1 Proxy Trajectories")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/gate_summary.png", bbox_inches="tight")
plt.show()
print(f"Figure saved: {OUT_DIR}/gate_summary.png")


## Download back (Kaggle → your machine)

From the **Output** tab (or `/kaggle/working/gate_output/`), download and place
under `archive/gate-results/` in the repo (gitignored):

- `all_results.pkl`
- `summary.json`
- `trajectories.csv`
- `*_run*.pkl` (per-run metrics)

Then run the analysis:

```bash
source hbar_env/bin/activate
python code/experiments/analyze_gate.py --results-dir archive/gate-results
```

which writes `paper/planning/gate-result.md` and applies the decision rule
(mechanistic / phenomenological / stop).
